In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = "../../helpers/lng_lnd"
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = "../../helpers/curated_helpers"
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_account_transaction_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "AccountTransactionId	string,"
        #     "SourceSystem	string,"
        #     "PortfolioKey	string,"
        #     "PortfolioName	string,"
        #     "IngestionTimestamp	timestamp,"
        #     "BatchId	string,"
        #     "AccountTransactionKey	string,"
        #     "DepositKey     string,"
        #     "ExposureKey	string,"
        #     "AccountType	string,"
        #     "TransactionTimeStamp	timestamp,"
        #     "TransactionINTSN	int,"
        #     "RequestSN	int,"
        #     "AccountCurrencyCode	string,"
        #     "AccountCurrencyDesc	string,"
        #     "JustificationCode	int,"
        #     "Justification	string,"
        #     "CurrencyCode	string,"
        #     "Currency	string,"
        #     "ChargeCurrencyCode	string,"
        #     "ChargeCurrency	string,"
        #     "TransactionCode	int,"
        #     "TransactionNumber	int,"
        #     "PenaltyInterestAmount	decimal(19,2),"
        #     "InterestAmount	decimal(19,2),"
        #     "CommisionAmount	decimal(19,2),"
        #     "ExpenseAmount	decimal(19,2),"
        #     "CapitalAmount	decimal(19,2),"
        #     "TransactionAmount	decimal(19,2),"
        #     "DormantAmount	decimal(19,2),"
        #     "BlockedAmount	decimal(19,2),"
        #     "UnclearAmount	decimal(19,2),"
        #     "PositiveAmount	decimal(19,2),"
        #     "UnrealizedInterestAmount	decimal(19,2),"
        #     "UnrealizedPenaltyInterestAmount	decimal(19,2),"
        #     "PUBUnrealizedInterestAmount	decimal(19,2),"
        #     "PUBAccruedInterestAmount	decimal(19,2),"
        #     "SubsidyAmount	decimal(19,2),"
        #     "AccruedInterestAmount	decimal(19,2),"
        #     "AccruedPenaltyInterestAmount	decimal(19,2),"
        #     "CCExpenseAmount	decimal(19,2),"
        #     "CCCommissionAmount	decimal(19,2),"
        #     "ConfirmedAmount	decimal(19,2),"
        #     "THRDReportingAmount	decimal(19,2),"
        #     "PublicCommisionAmount	decimal(19,2),"
        #     "ProductsAmount	decimal(19,2),"
        #     "InterestSpreadAmount	decimal(19,2),"
        #     "PreviousAccountingBalance	decimal(19,2),"
        #     "PreviousAccountBankBalance	decimal(19,2),"
        #     "PreviousOverdueAccountBalance	decimal(19,2),"
        #     "PreviousAccountBalance	decimal(19,2),"
        #     "AvailableDate	date,"
        #     "TransactionDate	date,"
        #     "CreatedDate	date,"
        #     "ExpirationDate	date,"
        #     "ValeurDate	date,"
        #     "ReversedFlag	string,"
        #     "EntryStatusCode	string,"
        #     "EntryStatus	string,"
        #     "RequestTypeCode	string,"
        #     "RequestType	string,"
        #     "RequestLoanStatusCode	string,"
        #     "RequestLoanStatus	string,"
        #     "TransactionUser	string,"
        #     "TransactionComments	string,"
        #     "TransactionTypeCode	int,"
        #     "TransactionType	string,"
        #     "DebitCreditFlag	string,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.AccountTransaction"

        # ================== Create Delta Table ==================

        # logging.info("Creating Delta Table...")
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")


        # ================== depositaccount ==================

        depositaccount_extra_filters = None

        if depositaccount_extra_filters is not None:
            depositaccount_extra_filters.append(date_filter)
            depositaccount_final_filters = " AND ".join(depositaccount_extra_filters)
        else:
            depositaccount_final_filters = date_filter

        depositaccount_selected_fields = [
            "account_number",
            "dep_open_unit",
            "dep_acc_number",
            "businessdate",
        ]

        depositaccount = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_depositaccount",
            depositaccount_final_filters,
            depositaccount_selected_fields,
        )

        # ================== portfolio ==================
        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "profits",
                )        
        else:
            logging.info("Tables doe not exist, no need to delete entries for business date!")


        # ================== loan_account_extrait ==================
        loan_account_extrait_extra_filters = None

        if loan_account_extrait_extra_filters is not None:
            loan_account_extrait_extra_filters.append(date_filter)
            loan_account_extrait_final_filters = " AND ".join(
                loan_account_extrait_extra_filters
            )
        else:
            loan_account_extrait_final_filters = date_filter

        loan_account_extrait_selected_fields = [
            "acc_unit",
            "acc_type",
            "acc_sn",
            "tmstamp",
            "trx_internal_sn",
            "trx_date",
            "request_sn",
            "acc_curr",
            "justification_code",
            "trx_curr",
            "chr_curr",
            "transaction_code",
            "trx_sn",
            "rl_pnl_int_amn",
            "rl_nrm_int_amn",
            "commission_amn",
            "expense_amn",
            "capital_amn",
            "trx_amn",
            "dormant_amn",
            "blocked_amn",
            "unclear_amn",
            "positive_amn",
            "url_nrm_int_amn",
            "url_pnl_int_amn",
            "url_pub_int_amn",
            "acr_pub_int_amn",
            "subsidy_amn",
            "acr_nrm_int_amn",
            "acr_pnl_int_amn",
            "exp_in_cc_amn",
            "com_in_cc_amn",
            "confirm_amn",
            "thrdprt_amn",
            "public_comm_amn",
            "products_amn",
            "int_spread_amn",
            "prv_account_bal",
            "prv_acc_bank_bal",
            "prv_ov_acc_bal",
            "prv_nrm_acc_bal",
            "avail_dt",
            "rq_creation_dt",
            "rq_expire_dt",
            "valeur_dt",
            "reversed_flg",
            "entry_status",
            "request_type",
            "request_loan_sts",
            "trx_usr",
            "extrait_comments",
            "gd_reason",
            "businessdate",
        ]

        loan_account_extrait = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_account_extrait",
            loan_account_extrait_final_filters,
            loan_account_extrait_selected_fields,
        )

        loan_account_extrait_primary_key_fields_1 = [
            loan_account_extrait.acc_unit,
            loan_account_extrait.acc_type,
            loan_account_extrait.acc_sn,
            loan_account_extrait.tmstamp,
            loan_account_extrait.trx_internal_sn,
            lit("profits"),
        ]

        loan_account_extrait_primary_key_fields_2 = [
            loan_account_extrait.acc_unit,
            loan_account_extrait.acc_type,
            loan_account_extrait.acc_sn,
            loan_account_extrait.tmstamp,
            loan_account_extrait.trx_internal_sn,
        ]

        loan_account_extrait = loan_account_extrait.withColumn(
            "AccountTransactionId",
            sha2(concat_ws(";", *loan_account_extrait_primary_key_fields_1), 256),
        ).withColumn(
            "AccountTransactionKey",
            concat_ws(";", *loan_account_extrait_primary_key_fields_2),
        )

        # ================== loan_details ==================
        loan_details_extra_filters = None

        if loan_details_extra_filters is not None:
            loan_details_extra_filters.append(date_filter)
            loan_details_final_filters = " AND ".join(loan_details_extra_filters)
        else:
            loan_details_final_filters = date_filter

        loan_details_selected_fields = [
            "prft_acc_unit",
            "account_number",
            "prft_acc_sn",
            "prft_acc_type",
        ]

        loan_details = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_details",
            loan_details_final_filters,
            loan_details_selected_fields,
        )

        # ================== deposittransaction ==================
        deposittransaction_extra_filters = None

        if deposittransaction_extra_filters is not None:
            deposittransaction_extra_filters.append(date_filter)
            deposittransaction_final_filters = " AND ".join(
                deposittransaction_extra_filters
            )
        else:
            deposittransaction_final_filters = date_filter

        deposittransaction_selected_fields = [
            "trans_ser_num",
            "entry_ser_num",
            "just_id",
            "trans_id",
            "amount",
            "trx_date",
            "value_date",
            "reverse_flag",
            "trx_usr",
            "entry_comments",
            "debit_credit_flag",
            "dep_number",
            "businessdate",
        ]

        deposittransaction = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_deposittransaction",
            deposittransaction_final_filters,
            deposittransaction_selected_fields,
        )

        # ======================== Params =====================

        params_extra_filters = None

        if params_extra_filters is not None:
            params_extra_filters.append(date_filter)
            params_final_filters = " AND ".join(params_extra_filters)
        else:
            params_final_filters = date_filter

        params_selected_fields = [
            "param_source",
            "param_type",
            "serial_num",
            "description",
        ]

        params = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_params",
            params_final_filters,
            params_selected_fields,
        )

        # Declaring a second table that will be used to join with loan_account_extrait but with different conditions
        params_identical = params
        params_identical = params_identical.withColumnRenamed(
            "param_type", "param_type_2"
        )
        params_identical = params_identical.withColumnRenamed(
            "param_source", "param_source_2"
        )
        params_identical = params_identical.withColumnRenamed(
            "description", "description_2"
        )

        # Conditions
        loan_account_extrait_loan_details_join_condition = (
            (loan_account_extrait.acc_unit == loan_details.prft_acc_unit)
            & (loan_account_extrait.acc_sn == loan_details.prft_acc_sn)
            & (loan_account_extrait.acc_type == loan_details.prft_acc_type)
        )

        loan_details_portfolio_join_condition = (
            portfolio.unit_code_profits == loan_details.prft_acc_unit
        )

        loan_account_extrait_params_join_condition_1 = (
            (loan_account_extrait.justification_code == params.serial_num)
            & (params.param_type == "JUSTIFIC")
            & (params.param_source == "JUSTIFIC")
        )

        loan_account_extrait_params_join_condition_2 = (
            (loan_account_extrait.gd_reason == params_identical.serial_num)
            & (params_identical.param_source_2 == "GD")
            & (params_identical.param_type_2 == "EXTYP")
        )

        # ================== Join (loan_account_extrait, loan_details, portfolio) ==================
        target_table = (
            loan_account_extrait.join(
                loan_details,
                loan_account_extrait_loan_details_join_condition,
                "left",
            )
            .join(
                portfolio,
                loan_details_portfolio_join_condition,
                "left",
            )
            .join(
                params,
                loan_account_extrait_params_join_condition_1,
                "left",
            )
            .join(
                params_identical, loan_account_extrait_params_join_condition_2, "left"
            )
        )

        target_table = (
            target_table.withColumn("DepositKey", lit(None).cast("string"))
            .withColumn(
                "RequestType",
                when(loan_account_extrait.request_type == "1", "Installment")
                .when(
                    loan_account_extrait.request_type == "2",
                    "Interest Request",
                )
                .when(loan_account_extrait.request_type == "3", "Loan Request")
                .when(loan_account_extrait.request_type == "4", "ExpenseRequest")
                .when(
                    loan_account_extrait.request_type == "5",
                    "CapitalRequest",
                )
                .otherwise(None),
            )
            .withColumn(
                "RequestLoanStatus",
                when(loan_account_extrait.request_loan_sts == "1", "Normal")
                .when(
                    loan_account_extrait.request_loan_sts == "2",
                    "Overdue",
                )
                .when(loan_account_extrait.request_loan_sts == "3", "Define Delay")
                .when(loan_account_extrait.request_loan_sts == "4", "Writen Off")
                .otherwise(None),
            )
        )

        # ================== Join (loan_details, portfolio, deposit_acount) ==================

        deposittransaction_primary_key_fields = [
            deposittransaction.trans_ser_num,
            deposittransaction.dep_number,
            lit("profits"),
        ]

        deposittransaction = (
            deposittransaction.withColumn(
                "AccountTransactionId",
                sha2(concat_ws(";", *deposittransaction_primary_key_fields), 256),
            )
            .withColumn(
                "AccountTransactionKey",
                concat_ws(
                    ";", deposittransaction.trans_ser_num, deposittransaction.dep_number
                ),
            )
            .withColumn("DepositKey", deposittransaction.dep_number)
        )

        depositaccount_deposittransaction_condition = (
            deposittransaction.dep_number == depositaccount.dep_acc_number
        )

        depositaccount_portfolio_join_condition = (
            portfolio.unit_code_profits == depositaccount.dep_open_unit
        )

        joined_df = deposittransaction.join(
            depositaccount, depositaccount_deposittransaction_condition, "inner"
        ).join(portfolio, depositaccount_portfolio_join_condition, "left")

        ### ================== account_transaction_profits_deposit_df ==================

        account_transaction_profits_deposit_df = joined_df.select(
            joined_df.AccountTransactionId.cast("string"),
            lit("profits").cast("String").alias("SourceSystem"),
            portfolio.unit_code_profits.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            joined_df.AccountTransactionKey.alias("AccountTransactionKey"),
            joined_df.DepositKey.cast("string").alias("DepositKey"),
            depositaccount.account_number.cast("string").alias("ExposureKey"),
            lit("Deposit").cast("String").alias("AccountType"),
            lit(None).cast("timestamp").alias("TransactionTimeStamp"),
            lit(None).cast("int").alias("TransactionINTSN"),
            joined_df.entry_ser_num.alias("RequestSN"),
            lit(None).cast("String").alias("AccountCurrencyCode"),
            lit(None).cast("String").alias("AccountCurrencyDesc"),
            joined_df.just_id.alias("JustificationCode"),
            lit(None).cast("string").alias("Justification"),
            lit(None).cast("String").alias("CurrencyCode"),
            lit(None).cast("String").alias("Currency"),
            lit(None).cast("String").alias("ChargeCurrencyCode"),
            lit(None).cast("String").alias("ChargeCurrency"),
            joined_df.trans_id.alias("TransactionCode"),
            lit(None).cast("int").alias("TransactionNumber"),
            lit(None).cast("decimal(19,2)").alias("PenaltyInterestAmount"),
            lit(None).cast("decimal(19,2)").alias("InterestAmount"),
            lit(None).cast("decimal(19,2)").alias("CommisionAmount"),
            lit(None).cast("decimal(19,2)").alias("ExpenseAmount"),
            lit(None).cast("decimal(19,2)").alias("CapitalAmount"),
            joined_df.amount.cast("decimal(19,2)").alias("TransactionAmount"),
            lit(None).cast("decimal(19,2)").alias("DormantAmount"),
            lit(None).cast("decimal(19,2)").alias("BlockedAmount"),
            lit(None).cast("decimal(19,2)").alias("UnclearAmount"),
            lit(None).cast("decimal(19,2)").alias("PositiveAmount"),
            lit(None).cast("decimal(19,2)").alias("UnrealizedInterestAmount"),
            lit(None).cast("decimal(19,2)").alias("UnrealizedPenaltyInterestAmount"),
            lit(None).cast("decimal(19,2)").alias("PUBUnrealizedInterestAmount"),
            lit(None).cast("decimal(19,2)").alias("PUBAccruedInterestAmount"),
            lit(None).cast("decimal(19,2)").alias("SubsidyAmount"),
            lit(None).cast("decimal(19,2)").alias("AccruedInterestAmount"),
            lit(None).cast("decimal(19,2)").alias("AccruedPenaltyInterestAmount"),
            lit(None).cast("decimal(19,2)").alias("CCExpenseAmount"),
            lit(None).cast("decimal(19,2)").alias("CCCommissionAmount"),
            lit(None).cast("decimal(19,2)").alias("ConfirmedAmount"),
            lit(None).cast("decimal(19,2)").alias("THRDReportingAmount"),
            lit(None).cast("decimal(19,2)").alias("PublicCommisionAmount"),
            lit(None).cast("decimal(19,2)").alias("ProductsAmount"),
            lit(None).cast("decimal(19,2)").alias("InterestSpreadAmount"),
            lit(None).cast("decimal(19,2)").alias("PreviousAccountingBalance"),
            lit(None).cast("decimal(19,2)").alias("PreviousAccountBankBalance"),
            lit(None).cast("decimal(19,2)").alias("PreviousOverdueAccountBalance"),
            lit(None).cast("decimal(19,2)").alias("PreviousAccountBalance"),
            lit(None).cast("date").alias("AvailableDate"),
            joined_df.trx_date.alias("TransactionDate"),
            lit(None).cast("date").alias("CreatedDate"),
            lit(None).cast("date").alias("ExpirationDate"),
            joined_df.value_date.alias("ValeurDate"),
            joined_df.reverse_flag.alias("ReversedFlag"),
            lit(None).cast("String").alias("EntryStatusCode"),
            lit(None).cast("String").alias("EntryStatus"),
            lit(None).cast("String").alias("RequestTypeCode"),
            lit(None).cast("String").alias("RequestType"),
            lit(None).cast("String").alias("RequestLoanStatusCode"),
            lit(None).cast("String").alias("RequestLoanStatus"),
            joined_df.trx_usr.alias("TransactionUser"),
            joined_df.entry_comments.alias("TransactionComments"),
            lit(None).cast("int").alias("TransactionTypeCode"),
            lit(None).cast("string").alias("TransactionType"),
            joined_df.debit_credit_flag.alias("DebitCreditFlag"),
            depositaccount.businessdate.alias("BusinessDate"),
        )

        # ================== account_transaction_profits_other_df ==================

        account_transaction_profits_other_df = target_table.select(
            target_table.AccountTransactionId.cast("string"),
            lit("profits").cast("String").alias("SourceSystem"),
            loan_details.prft_acc_unit.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            target_table.AccountTransactionKey.cast("String"),
            target_table.DepositKey.alias("DepositKey"),
            loan_details.account_number.alias("ExposureKey"),
            lit("exposure").cast("string").alias("AccountType"),
            target_table.tmstamp.alias("TransactionTimeStamp"),
            target_table.trx_internal_sn.alias("TransactionINTSN"),
            target_table.request_sn.alias("RequestSN"),
            target_table.acc_curr.alias("AccountCurrencyCode"),
            target_table.acc_curr.alias("AccountCurrencyDesc"),
            target_table.justification_code.alias("JustificationCode"),
            target_table.description.cast("string").alias("Justification"),
            target_table.trx_curr.alias("CurrencyCode"),
            target_table.trx_curr.alias("Currency"),
            target_table.chr_curr.alias("ChargeCurrencyCode"),
            target_table.chr_curr.alias("ChargeCurrency"),
            target_table.transaction_code.alias("TransactionCode"),
            target_table.trx_sn.alias("TransactionNumber"),
            target_table.rl_pnl_int_amn.alias("PenaltyInterestAmount"),
            target_table.url_nrm_int_amn.alias("InterestAmount"),
            target_table.commission_amn.alias("CommisionAmount"),
            target_table.expense_amn.alias("ExpenseAmount"),
            target_table.capital_amn.alias("CapitalAmount"),
            target_table.trx_amn.alias("TransactionAmount"),
            target_table.dormant_amn.alias("DormantAmount"),
            target_table.blocked_amn.alias("BlockedAmount"),
            target_table.unclear_amn.alias("UnclearAmount"),
            target_table.positive_amn.alias("PositiveAmount"),
            target_table.url_nrm_int_amn.alias("UnrealizedInterestAmount"),
            target_table.url_pnl_int_amn.alias("UnrealizedPenaltyInterestAmount"),
            target_table.url_pub_int_amn.alias("PUBUnrealizedInterestAmount"),
            target_table.acr_pub_int_amn.alias("PUBAccruedInterestAmount"),
            target_table.subsidy_amn.alias("SubsidyAmount"),
            target_table.acr_nrm_int_amn.alias("AccruedInterestAmount"),
            target_table.acr_pnl_int_amn.alias("AccruedPenaltyInterestAmount"),
            target_table.exp_in_cc_amn.alias("CCExpenseAmount"),
            target_table.com_in_cc_amn.alias("CCCommissionAmount"),
            target_table.confirm_amn.alias("ConfirmedAmount"),
            target_table.thrdprt_amn.alias("THRDReportingAmount"),
            target_table.public_comm_amn.alias("PublicCommisionAmount"),
            target_table.products_amn.alias("ProductsAmount"),
            target_table.int_spread_amn.alias("InterestSpreadAmount"),
            target_table.prv_account_bal.alias("PreviousAccountingBalance"),
            target_table.prv_acc_bank_bal.alias("PreviousAccountBankBalance"),
            target_table.prv_ov_acc_bal.alias("PreviousOverdueAccountBalance"),
            target_table.prv_nrm_acc_bal.alias("PreviousAccountBalance"),
            target_table.avail_dt.alias("AvailableDate"),
            target_table.trx_date.alias("TransactionDate"),
            loan_account_extrait.rq_creation_dt.alias("CreatedDate"),
            target_table.rq_expire_dt.alias("ExpirationDate"),
            target_table.valeur_dt.alias("ValeurDate"),
            target_table.reversed_flg.alias("ReversedFlag"),
            target_table.entry_status.alias("EntryStatusCode"),
            target_table.entry_status.alias("EntryStatus"),
            target_table.request_type.alias("RequestTypeCode"),
            target_table.RequestType.cast("string").alias("RequestType"),
            target_table.request_loan_sts.alias("RequestLoanStatusCode"),
            target_table.RequestLoanStatus.cast("string").alias("RequestLoanStatus"),
            target_table.trx_usr.alias("TransactionUser"),
            target_table.extrait_comments.alias("TransactionComments"),
            target_table.gd_reason.alias("TransactionTypeCode"),
            target_table.description_2.cast("string").alias("TransactionType"),
            lit(None).cast("String").alias("DebitCreditFlag"),
            loan_account_extrait.businessdate.alias("BusinessDate"),
        )

        # ================== Selection of Final Columns ==================
        # ================== union all ==================

        result_df = account_transaction_profits_deposit_df.unionAll(
            account_transaction_profits_other_df
        )

        # ================== Append ==================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")



In [0]:
load_curated_account_transaction_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)
